In [3]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
print(X)
net(X)

tensor([[0.6452, 0.4179, 0.6675, 0.2628, 0.1662, 0.8187, 0.3963, 0.6167, 0.4776,
         0.5103, 0.0907, 0.0531, 0.9907, 0.7666, 0.3455, 0.8348, 0.5894, 0.8083,
         0.1789, 0.3779],
        [0.3643, 0.5531, 0.5071, 0.6058, 0.8054, 0.8120, 0.6260, 0.4674, 0.4705,
         0.1658, 0.0192, 0.1412, 0.9716, 0.8351, 0.4475, 0.3298, 0.6741, 0.1761,
         0.2532, 0.3117]])


tensor([[-0.0991, -0.1141,  0.2926,  0.0248,  0.0627,  0.0676, -0.3109, -0.0553,
          0.0370, -0.3434],
        [-0.1114, -0.1220,  0.1895,  0.0565,  0.0040, -0.0398, -0.2828, -0.1529,
          0.0134, -0.3342]], grad_fn=<AddmmBackward0>)

In [4]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义。
        return self.out(F.relu(self.hidden(X)))

In [5]:
net = MLP()
net(X)

tensor([[ 0.3137, -0.0161, -0.0686,  0.0543,  0.1589, -0.1698, -0.0158,  0.1499,
          0.1370,  0.1424],
        [ 0.2915,  0.0372, -0.1068,  0.0202,  0.2530, -0.1348,  0.0694,  0.0806,
          0.2417, -0.0240]], grad_fn=<AddmmBackward0>)

enumerate() 会同时提供编号和对应的网络层。
对于：
args = (
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)
循环过程大致是：
idx=0，module=Linear(20→256)
idx=1，module=ReLU()
idx=2，module=Linear(256→10)
其中：
idx       当前层的编号
module    当前网络层对象
5. 把网络层登记到 _modules
self._modules[str(idx)] = module
每一层以字符串编号作为名字：
self._modules = {
    "0": nn.Linear(20, 256),
    "1": nn.ReLU(),
    "2": nn.Linear(256, 10)
}
这里不只是把网络层放进普通字典。登记到 _modules 后，PyTorch才能知道这些层属于当前网络。
例如第一层 Linear 里面有权重和偏置：
weight
bias
因为它被登记在 _modules 中，所以这些参数会自动出现在：
net.parameters()
如果只是随便存进某些普通容器，PyTorch不一定能发现和管理其中的参数。
实际项目中一般不建议直接操作 _modules，更规范的写法是：
self.add_module(str(idx), module)
这里直接写 _modules 是为了展示 nn.Module 的内部工作原理。

7. 依次经过所有层
for block in self._modules.values():
    X = block(X)
self._modules.values() 按照添加顺序取出网络层。
假设初始输入为：
X.shape == (32, 20)
这里 32 是批量大小，20 是每个样本的特征数。
执行过程如下：
X = Linear(20, 256)(X)
形状变成：
(32, 20) → (32, 256)
接着：
X = ReLU()(X)
形状不变，但负数会变成 0：
(32, 256) → (32, 256)
最后：
X = Linear(256, 10)(X)
形状变成：
(32, 256) → (32, 10)
注意每次都把处理后的结果重新赋给 X：
原始 X
  ↓ 第一层
新的 X
  ↓ 第二层
新的 X
  ↓ 第三层
最终 X

In [6]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

In [7]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [8]:
net = FixedHiddenMLP()
net(X)

tensor(-0.4767, grad_fn=<SumBackward0>)

In [9]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(0.2790, grad_fn=<SumBackward0>)